# Red neuronal multicapa (MLP) con MNIST

**Materia:** Inteligencia Artificial  
**Profesor:** José Mario Ríos Félix  
**Horario:** 6:00 a 7:00 p. m.  
**Alumno:** Jonathan Daniel Valencia  
**Número de control:** 23170373  

## Descripción

Esta aplicación carga el modelo MLP guardado en `mlp_model.keras`, entrenado previamente con el dataset MNIST en el notebook original, y lo prueba en tiempo real. El usuario dibuja un dígito del 0 al 9 en un lienzo y la aplicación muestra las tres clases con mayor probabilidad. El modelo no se vuelve a entrenar.

## Librerías

Se importan TensorFlow (para cargar el modelo), NumPy (arreglos), OpenCV (procesamiento de imágenes) y Gradio (interfaz web con lienzo de dibujo).

In [1]:
import tensorflow as tf  # Carga del modelo entrenado (Keras)
import numpy as np       # Manejo de arreglos numéricos
import cv2               # Procesamiento de imágenes (OpenCV)
import gradio as gr      # Interfaz web con lienzo de dibujo

## 1. Cargar el modelo y preparar la imagen

El modelo se entrenó con imágenes de MNIST: 28x28 píxeles, en escala de grises, con el dígito en blanco sobre fondo negro, de unos 20 píxeles de alto y centrado. El lienzo entrega un dibujo de 280x280 con fondo blanco, por lo que la función `clasificar_trazo` lo adapta antes de predecir:

1. Extrae la imagen del diccionario que entrega Gradio.
2. Invierte los colores para obtener fondo negro y trazo blanco.
3. Recorta el dígito y lo escala a 20 píxeles en su lado mayor, conservando la proporción.
4. Lo coloca en un lienzo de 28x28 y lo desplaza para que su centro de masa quede en el centro, como en MNIST.
5. Normaliza los valores a 0-1, igual que en el entrenamiento, y agrega la dimensión de lote.
6. Calcula la probabilidad de cada clase con `model.predict`.

Si el lienzo está vacío, la función muestra un aviso en lugar de predecir.

In [2]:
# 1. Cargar el modelo entrenado desde el archivo .keras
model = tf.keras.models.load_model("mlp_model.keras")

def clasificar_trazo(img):
    # Adaptar la imagen de entrada al formato del modelo MLP entrenado en MNIST (28x28 píxeles, escala de grises)

    # Gradio entrega un diccionario; el dibujo está en la clave "composite" como un array numpy de 280x280 píxeles
    # (un solo canal de grises, gracias a image_mode="L" en el lienzo)
    if isinstance(img, dict):
        img = img["composite"]

    # MNIST usa dígito blanco sobre fondo negro; si el lienzo es blanco, se invierte
    if img[0, 0] > 128:
        img = cv2.bitwise_not(img)

    # Si el lienzo está vacío no hay dígito que clasificar
    puntos = cv2.findNonZero(img)
    if puntos is None:
        gr.Warning("Dibuja un dígito antes de enviar.")
        return None

    # Recortar el número dejando solo el rectángulo que lo contiene, escalarlo a 20 px en su lado mayor y centrarlo en 28x28 (formato de MNIST)
    x, y, w, h = cv2.boundingRect(puntos)
    recorte = img[y:y+h, x:x+w]

    # Se calcula el tamaño al que se reducirá el número: su lado más largo medirá 20 píxeles y el otro se
    # ajusta en proporción, para que el número no se deforme.
    # - 20 (valor): tamaño de la caja de 20 x 20 píxeles en la que MNIST colocó cada dígito
    # - nuevo_w, nuevo_h (variables): nuevo ancho y nuevo alto
    # - h * (20 / w) (fórmula): regla de tres que conserva la proporción; int() quita los decimales
    # - if / else (condicional): decide cuál de los dos lados es el más largo
    if w > h:
        nuevo_w = 20
        nuevo_h = int(h * (20 / w))
    else:
        nuevo_h = 20
        nuevo_w = int(w * (20 / h))

    # Se reduce el recorte a ese tamaño.
    recorte_resized = cv2.resize(recorte, (max(1, nuevo_w), max(1, nuevo_h)),
                                 interpolation=cv2.INTER_AREA)

    # Se prepara un cuadro negro de 28 x 28 (el tamaño que espera el modelo) y se pega el número en el centro.
    img_final = np.zeros((28, 28), dtype=np.uint8)
    inicio_x = (28 - nuevo_w) // 2
    inicio_y = (28 - nuevo_h) // 2
    img_final[inicio_y:inicio_y+nuevo_h, inicio_x:inicio_x+nuevo_w] = recorte_resized

    # Desplazar la imagen para que el centro de masa quede en (14, 14), como en MNIST
    m = cv2.moments(img_final)
    cx, cy = m["m10"] / m["m00"], m["m01"] / m["m00"]
    M = np.float32([[1, 0, 14 - cx], [0, 1, 14 - cy]])
    img_final = cv2.warpAffine(img_final, M, (28, 28))

    # Normalizar a 0-1 y agregar la dimensión de lote: (1, 28, 28)
    img_normalized = img_final.astype("float32") / 255.0
    img_input = np.expand_dims(img_normalized, axis=0)

    # Calcular la probabilidad de cada clase (0 al 9)
    predicciones = model.predict(img_input, verbose=0)[0]

    return {str(i): float(predicciones[i]) for i in range(10)}

## 2. Crear la interfaz

Se usa Gradio para construir una interfaz web sencilla. `Sketchpad` es el lienzo de dibujo (280x280, imagen en escala de grises) y `Label` muestra las tres clases más probables con su porcentaje. El parámetro `flagging_mode="never"` desactiva el botón Flag, que no interviene en el reconocimiento.

In [3]:
# 2. Crear la interfaz web: lienzo de dibujo como entrada y las 3 clases más probables como salida
interfaz = gr.Interface(
    fn=clasificar_trazo,
    inputs=gr.Sketchpad(canvas_size=(280, 280), type="numpy", image_mode="L"),
    outputs=gr.Label(num_top_classes=3),
    title="Reconocimiento de Dígitos MNIST - Multilayer Perceptron",
    description="Dibuja un dígito del 0 al 9 en el lienzo para analizar la predicción de la red neuronal.",
    flagging_mode="never",
)

## 3. Ejecutar la aplicación

`launch()` inicia un servidor local. La interfaz se muestra dentro del notebook y también en la URL que aparece en la salida de la celda.

In [5]:
# 3. Iniciar la aplicación en un servidor local; la interfaz se muestra en el notebook y en la URL indicada
interfaz.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.


## Observaciones

- El modelo alcanzó una precisión aproximada de 94 % en el conjunto de prueba (notebook original), por lo que no acierta siempre.
- Un MLP es sensible a la posición y al estilo del trazo. Por eso la imagen se recorta, se escala y se centra antes de predecir.
- La salida siempre reparte el 100 % entre las 10 clases. Con una imagen vacía el modelo elegiría una clase arbitraria, por lo que la función lo evita.
- Algunos estilos de trazo generan menos confianza. Por ejemplo, un 4 de punta cerrada obtuvo 66 % y el 9 recibió 28 %.